[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HesusG/dammg-supervisado/blob/main/notebooks/salas/S2_en_vivo.ipynb)

# S2 en vivo: métricas, columnas nuevas y regresión logística

Notebook del profesor. Dos tipos de celda:

- **▶ Correr**: ya está escrita; córrela y explica qué sale.
- **⌨ Teclear**: una o dos líneas para escribir en vivo. El código ya está escrito en la celda por si te trabas:
  bórralo y tecléalo, o córrelo tal cual. El comentario dice qué debe salir.

Corre las celdas en orden. Si Colab se reinicia, vuelve a correr desde la preparación.

In [ ]:
# @title Preparación del entorno: solo córrela, no hace falta leerla
# Descarga los datos y dammg_utils.py, revisa que no lleguen dañados y avisa con ✅ entorno listo.
import hashlib, importlib.util, os, pathlib, subprocess, sys, urllib.request

BASE_URL = "https://hesusg.github.io/dammg-supervisado/datos/"
# Versiones fijas: copia generada de notebooks/requirements.txt (fuente única)
REQUISITOS = {
    'numpy': 'numpy==2.5.4',
    'pandas': 'pandas==3.0.6',
    'sklearn': 'scikit-learn==1.9.1',
    'matplotlib': 'matplotlib==3.11.2',
    'ipywidgets': 'ipywidgets==8.1.9',
    'pyarrow': 'pyarrow==26.0.0',
}
SHA256 = {
    'mibici_curso.parquet': 'a5fd844c57bbf220f0ddfc27dc965ef8fb8438bd6c1167709bde7ccc1dafa6d8',
    'dammg_utils.py': '7499b8695cb94d4dd72133e197d95a143e8b05b82f1d91928f6a8a30fa0f62d0',
}
EN_EVALUADOR = os.environ.get("DAMMG_GRADER") == "1"
# Umbral del evento |future_net| (percentil 85 de train), congelado para todo el curso
UMBRAL_EVENTO = 7.0


def _sha256(ruta):
    return hashlib.sha256(pathlib.Path(ruta).read_bytes()).hexdigest()


def _copia_local(nombre):
    """Busca una copia local: DAMMG_DATA_DIR o el repo del curso (data/curso, notebooks)."""
    carpetas = []
    if os.environ.get("DAMMG_DATA_DIR"):
        carpetas.append(pathlib.Path(os.environ["DAMMG_DATA_DIR"]))
    aqui = pathlib.Path.cwd().resolve()
    for p in [aqui, *aqui.parents][:5]:
        carpetas += [p / "data" / "curso", p / "notebooks", p]
    for c in carpetas:
        if (c / nombre).is_file():
            return c / nombre
    return None


def _obtener(nombre, esperado):
    """Copia local si existe; si no, descarga del sitio del curso y verifica el sha256."""
    if EN_EVALUADOR:
        # Dentro del evaluador: nada de descargar; los datos ya están en /data
        return pathlib.Path("/data") / nombre
    local = _copia_local(nombre)
    if local is not None:
        if _sha256(local) != esperado:
            print(f"⚠ {nombre}: la copia local no coincide con el sha256 publicado (¿versión en desarrollo?)")
        return local
    destino = pathlib.Path("datos_dammg") / nombre
    destino.parent.mkdir(exist_ok=True)
    if not destino.exists() or _sha256(destino) != esperado:
        print(f"Descargando {nombre} ...")
        try:
            urllib.request.urlretrieve(BASE_URL + nombre, destino)
        except OSError as e:   # HTTPError, sin internet, etc.
            raise RuntimeError(
                f"No pude descargar {nombre} ({e}). Revisa tu conexión y vuelve a ejecutar esta celda. "
                "Si sigue fallando, avisa en el foro de Moodle: puede que el archivo aún no esté publicado."
            ) from None
    assert _sha256(destino) == esperado, f"{nombre} llegó incompleto o alterado: bórralo y vuelve a ejecutar."
    return destino


if not EN_EVALUADOR:
    faltan = [pin for modulo, pin in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
    if faltan:
        print("Instalando:", ", ".join(faltan))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
RUTAS = {nombre: _obtener(nombre, esperado) for nombre, esperado in SHA256.items()}

sys.path.insert(0, str(RUTAS["dammg_utils.py"].parent))
RUTA_DATOS = RUTAS["mibici_curso.parquet"]

import dammg_utils as du
du.entorno_listo()

## Sección 1. Las preguntas de ayer: métricas de regresión (9:10 a 10:10)

### 1.1 Los datos de ayer

In [ ]:
# ▶ CORRER
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score, mean_absolute_percentage_error

df = pd.read_parquet(RUTA_DATOS)                 # una fila = una estación en una hora
train = df[df.particion == "train"]              # octubre a junio: para ajustar
julio = df[df.particion == "val"]                # julio: validación, para medir y escoger
print(f"train: {len(train):,} filas   julio: {len(julio):,} filas")   # debe salir: train: 254,440 filas   julio: 29,760 filas

In [ ]:
# ▶ CORRER
# La recta de ayer: salidas de hoy a partir de las de hace una semana, ajustada con train
recta = LinearRegression().fit(train[["departures_lag_168h"]], train.departures)
print(f"w = {recta.coef_[0]:.2f}   b = {recta.intercept_:.2f}")   # debe salir: w = 0.70   b = 1.18

### 1.2 Cinco horas reales: estación 51, miércoles 1 de julio, de 6:00 a 10:00

In [ ]:
# ▶ CORRER
# departures es la y: cuántas bicis salieron de verdad. La regla dice: las mismas que hace una semana
cinco = julio[(julio.station_id == 51) & (julio.hour >= "2026-07-01 06:00") & (julio.hour <= "2026-07-01 10:00")]
cinco = cinco[["hour_of_day", "departures"]].assign(regla=cinco.departures_lag_168h)
cinco                                            # debe salir: real 11 30 25 25 11; regla 12 30 17 20 17

In [ ]:
# ⌨ TECLEAR EN VIVO (o córrela tal cual)
cinco["error"] = cinco.regla - cinco.departures   # error = predicción menos real (ŷ − y)
cinco                                            # debe salir: 1, 0, -8, -5, 6

### 1.3 Del residuo a un solo número: SSE, MSE, RMSE y MAE

In [ ]:
# ▶ CORRER
sse = (cinco.error ** 2).sum()                   # SSE: suma de los errores al cuadrado
mse = (cinco.error ** 2).mean()                  # MSE: su promedio
print(f"SSE: {sse:.1f}   MSE: {mse:.1f}")        # debe salir: SSE: 126.0   MSE: 25.2

In [ ]:
# ⌨ TECLEAR EN VIVO (o córrela tal cual)
mse ** 0.5                                       # RMSE: raíz del MSE, de vuelta a bicis. Debe salir: np.float32(5.0199604), o sea 5.02

In [ ]:
# ⌨ TECLEAR EN VIVO (o córrela tal cual)
cinco.error.abs().mean()                         # MAE: promedio del error sin signo. Debe salir: np.float32(4.0)

In [ ]:
# ▶ CORRER
# Lo mismo con todo julio y con scikit-learn, para la regla y para la recta
y = julio.departures
modelos = {"regla": julio.departures_lag_168h, "recta": recta.predict(julio[["departures_lag_168h"]])}
pd.DataFrame({n: {"MAE": mean_absolute_error(y, p), "RMSE": root_mean_squared_error(y, p),
                  "MSE": root_mean_squared_error(y, p) ** 2, "J = MSE/2": root_mean_squared_error(y, p) ** 2 / 2}
              for n, p in modelos.items()}).round(2)
# debe salir: regla MAE 2.30, RMSE 4.16, MSE 17.31, J 8.65; recta MAE 2.35, RMSE 3.84, MSE 14.73, J 7.36

### 1.4 MAE contra RMSE: los modelos A y B

In [ ]:
# ▶ CORRER
# Cinco horas de juguete: A acierta cuatro y falla por 10 en el pico; B falla siempre por 3
real = [0, 3, 8, 12, 25]
A = [0, 3, 8, 12, 15]
B = [3, 6, 5, 15, 22]
for nombre, pred in {"A": A, "B": B}.items():
    print(f"{nombre}: MAE {mean_absolute_error(real, pred):.2f}   RMSE {root_mean_squared_error(real, pred):.2f}")
# debe salir: A: MAE 2.00   RMSE 4.47 / B: MAE 3.00   RMSE 3.00

### 1.5 Correlación de Pearson y R²

In [ ]:
# ⌨ TECLEAR EN VIVO (o córrela tal cual)
train[["departures", "departures_lag_168h"]].corr()   # debe salir: 0.70 fuera de la diagonal

In [ ]:
# ▶ CORRER
# En la recta de una sola columna, R² en train es r al cuadrado
r = train[["departures", "departures_lag_168h"]].corr().iloc[0, 1]
r2 = r2_score(train.departures, recta.predict(train[["departures_lag_168h"]]))
print(f"r al cuadrado: {r ** 2:.3f}   R² train: {r2:.3f}")   # debe salir: 0.494 y 0.494

In [ ]:
# ▶ CORRER
# R² en julio: compara contra decir siempre el promedio de julio
for n, p in modelos.items():
    print(f"R² julio, {n}: {r2_score(y, p):.2f}")   # debe salir: regla 0.41 / recta 0.50

### 1.6 MAPE con ceros

In [ ]:
# ▶ CORRER
print((y == 0).mean().round(3))                  # horas de julio con 0 salidas. Debe salir: 0.288
print(mean_absolute_percentage_error(y, modelos["regla"]))   # debe salir: 609105819467776.0 (6 × 10^14), un número absurdo

## Sección 2. Características derivadas y la fuga (10:20 a 11:20)

### 2.1 Los datos

In [ ]:
# ▶ CORRER
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

df = pd.read_parquet(RUTA_DATOS)                      # una fila = una estación en una hora
df = df.sort_values(["station_id", "hour"])           # por estación y, dentro de cada una, por hora
print(df.shape)                                       # sale: (342680, 19)

### 2.2 Calendario: ¿es hora pico?

In [ ]:
# ⌨ TECLEAR EN VIVO (o córrela tal cual)
df["pico"] = df.hour_of_day.isin([7, 8, 9, 17, 18, 19])   # True en las horas de entrada y salida del trabajo
# sale: nada (solo crea la columna)

In [ ]:
# ▶ CORRER
# Salidas promedio por hora, fuera y dentro de la hora pico
df.groupby("pico").departures.mean().round(2)          # sale: False 3.21, True 6.65

### 2.3 Rezago: lo que pasó ayer a esta misma hora

In [ ]:
# ⌨ TECLEAR EN VIVO (o córrela tal cual)
df["mi_rezago"] = df.groupby("station_id").departures.shift(24)   # 24 filas atrás, dentro de cada estación
# sale: nada (solo crea la columna)

In [ ]:
# ▶ CORRER
# ¿Coincide con la columna que ya traen los datos?
iguales = (df.mi_rezago == df.departures_lag_24h).sum()
vacias = df.mi_rezago.isna().sum()                    # el primer día de cada estación no tiene "ayer"
print(f"coinciden: {iguales:,}   vacías: {vacias}")   # sale: coinciden: 341,720   vacías: 960

In [ ]:
# ▶ CORRER
# Revisión a mano: la estación 51 a las 8:00, dos días seguidos de julio
dias = pd.to_datetime(["2026-07-09 08:00", "2026-07-10 08:00"])
df[(df.station_id == 51) & df.hour.isin(dias)][["hour", "departures", "mi_rezago"]]
# sale: el 9 de julio, 35 salidas (rezago 29.0); el 10 de julio, 33 salidas y su rezago dice 35.0

### 2.4 Por grupo: el promedio de la estación a esa hora y tipo de día, calculado solo con train

In [ ]:
# ⌨ TECLEAR EN VIVO (o córrela tal cual)
tabla = df[df.particion == "train"].groupby(["station_id", "hour_of_day", "is_weekend"]).departures.mean()
# sale: nada (solo crea la tabla)

In [ ]:
# ▶ CORRER
# Lo que el encargado de la 51 sabe de memoria: a las 8, a las 13 y a las 18 (0 = entre semana, 1 = fin de semana)
tabla.loc[51].unstack().loc[[8, 13, 18]].round(1)      # sale: 8 h 29.0 y 12.8; 13 h 11.4 y 17.0; 18 h 19.5 y 8.5

In [ ]:
# ▶ CORRER
# La tabla como columna nueva: cada fila recibe el promedio de su estación a su hora
df = df.join(tabla.rename("prom_est_hora"), on=["station_id", "hour_of_day", "is_weekend"])
julio = df[df.particion == "val"]
mae = mean_absolute_error(julio.departures, julio.prom_est_hora)
print(f"MAE julio: {mae:.2f}")                       # sale: MAE julio: 1.95, la tabla de S1

### 2.5 Interacción y MAE con y sin cada columna

In [ ]:
# ▶ CORRER
# Interacción: una etiqueta por cada estación, hora y tipo de día (entre semana o fin de semana)
df["est_hora_finde"] = df.station_id.astype(str) + "_" + df.hour_of_day.astype(str) + "_" + df.is_weekend.astype(str)
train = df[df.particion == "train"]                   # octubre a junio: para ajustar
julio = df[df.particion == "val"]                     # julio: para medir

In [ ]:
# ▶ CORRER
def prueba(columnas, categorias=()):
    """Ajusta la recta con train y dice su MAE en julio. Las de `categorias` se vuelven columnas de 0 y 1."""
    prep = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), list(categorias)),
                              ("num", "passthrough", [c for c in columnas if c not in categorias])])
    modelo = make_pipeline(prep, LinearRegression()).fit(train[columnas], train.departures)
    return round(mean_absolute_error(julio.departures, modelo.predict(julio[columnas])), 2)

In [ ]:
# ▶ CORRER
rezagos = ["departures_lag_24h", "departures_lag_168h"]
print(f"MAE julio, solo rezagos:       {prueba(rezagos):.2f}")                                    # sale: 2.13
print(f"MAE julio, + pico:             {prueba(rezagos + ['pico']):.2f}")                         # sale: 2.11
print(f"MAE julio, + hora:             {prueba(rezagos + ['hour_of_day'], ['hour_of_day']):.2f}")   # sale: 1.99
print(f"MAE julio, + prom_est_hora:    {prueba(rezagos + ['prom_est_hora']):.2f}")                # sale: 1.86
print(f"MAE julio, + est × hora × día: {prueba(rezagos + ['est_hora_finde'], ['est_hora_finde']):.2f}")   # sale: 1.85

### 2.6 La fuga: una columna que a las 7:59 todavía no existe

In [ ]:
# ▶ CORRER
# arrivals = las bicis que LLEGAN en esa misma hora, de 8 a 9. A las 7:59 nadie lo sabe.
honesto = rezagos + ["est_hora_finde"]
print(f"MAE julio, honesto:    {prueba(honesto, ['est_hora_finde']):.2f}")                  # sale: 1.85
print(f"MAE julio, + llegadas: {prueba(honesto + ['arrivals'], ['est_hora_finde']):.2f}")   # sale: 1.43

In [ ]:
# ▶ CORRER
# Y si además le doy net_flow (salidas menos llegadas de esa hora), la recta solo tiene que sumar:
print(f"MAE julio, + llegadas y neto: {prueba(honesto + ['arrivals', 'net_flow'], ['est_hora_finde']):.2f}")  # sale: 0.00

## Sección 3. Regresión logística y métricas de clasificación (11:45 a 12:50)

### 3.1 Los datos: la etiqueta de sí o no

In [ ]:
# ▶ CORRER
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, log_loss, precision_score, recall_score

df = pd.read_parquet(RUTA_DATOS)
train = df[df.particion == "train"].copy()   # octubre a junio: para ajustar
julio = df[df.particion == "val"].copy()     # julio: para medir
for d in (train, julio):
    d["y"] = (d.future_net.abs() >= 7).astype(int)   # 1 = desbalance fuerte en las próximas 3 horas
    d["x"] = d.net_lag_168h.abs()                     # desbalance de esa hora, hace una semana
print(f"train: {train.y.mean():.1%} de horas con desbalance   julio: {julio.y.mean():.1%}")
# sale: train: 15.9% de horas con desbalance   julio: 14.5%

### 3.2 La curva en S con una variable

In [ ]:
# ⌨ TECLEAR EN VIVO (o córrela tal cual)
m1 = LogisticRegression().fit(train[["x"]], train.y)   # busca la curva en S que mejor explica train
# sale: nada (tarda 1 o 2 segundos)

In [ ]:
# ▶ CORRER
# coef_ es qué tan empinada es la curva; intercept_ la recorre a la izquierda o a la derecha
print(f"w = {m1.coef_[0][0]:.2f}   b = {m1.intercept_[0]:.2f}")
# sale: w = 0.28   b = -2.30

In [ ]:
# ▶ CORRER
# Tres horas nuevas: poco, mediano y mucho desbalance hace una semana
nuevas = pd.DataFrame({"x": [0, 8.2, 20]})
m1.predict_proba(nuevas)[:, 1].round(2)   # columna 1 = probabilidad de desbalance
# sale: array([0.09, 0.5 , 0.96])

### 3.3 Modelos simples y complicados: ¿sirve el día par?

In [ ]:
# ▶ CORRER
# El modelo con desbalance, hora y estación (hora y estación en one-hot), con y sin el día par
for d in (train, julio):
    d["dia_par"] = (d.hour.dt.day % 2 == 0).astype(int)                      # 1 = día del mes par
def cols(d):   # x, día par, la hora en one-hot y la estación en one-hot
    return pd.concat([d[["x", "dia_par"]], pd.get_dummies(d.hour_of_day, prefix="h", dtype=int),
                      pd.get_dummies(d.station_id, prefix="e", dtype=int)], axis=1)
# sale: nada (solo prepara las columnas)

In [ ]:
# ▶ CORRER
sin_par = LogisticRegression(max_iter=1000).fit(cols(train).drop(columns="dia_par"), train.y)
con_par = LogisticRegression(max_iter=1000).fit(cols(train), train.y)
print(f"peso del día par: {con_par.coef_[0][1]:.2f}")
print(f"log-loss julio sin: {log_loss(julio.y, sin_par.predict_proba(cols(julio).drop(columns='dia_par'))[:, 1]):.3f}"
      f"   con: {log_loss(julio.y, con_par.predict_proba(cols(julio))[:, 1]):.3f}")
# sale (unos 15 segundos): peso del día par: 0.02 / log-loss julio sin: 0.317   con: 0.317

### 3.4 Las métricas con julio (el modelo de 12 columnas del Lab 2)

In [ ]:
# ▶ CORRER
# El modelo del Lab 2, ya escrito: 12 columnas (rezagos, promedios, capacidad, fin de semana, hora pico)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
def doce(d):
    return pd.DataFrame({"net_lag_24h": d.net_lag_24h, "net_lag_168h": d.net_lag_168h, "net_mean_24h": d.net_mean_24h,
        "abs_net_lag_24h": d.net_lag_24h.abs(), "abs_net_lag_168h": d.net_lag_168h.abs(), "abs_net_mean_24h": d.net_mean_24h.abs(),
        "departures_mean_24h": d.departures_mean_24h, "departures_mean_168h": d.departures_mean_168h,
        "departures_lag_168h": d.departures_lag_168h, "capacity": d.capacity, "is_weekend": d.is_weekend,
        "pico": d.hour_of_day.isin([7, 8, 9, 17, 18, 19, 20]).astype(int)})
modelo = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(doce(train), train.y)
p_julio = modelo.predict_proba(doce(julio))[:, 1]   # p̂ de cada hora de julio
# sale: nada (tarda unos 3 segundos)

In [ ]:
# ⌨ TECLEAR EN VIVO (o córrela tal cual)
alerta = (p_julio >= 0.5).astype(int)   # 1 = mando camión
# sale: nada

In [ ]:
# ⌨ TECLEAR EN VIVO (o córrela tal cual)
confusion_matrix(julio.y, alerta).ravel()   # primero lo que pasó, luego lo que dijo: TN, FP, FN, TP
# sale: array([24975,   458,  3519,   808])

In [ ]:
# ▶ CORRER
# precision: de mis alertas, cuántas eran ciertas. recall: de los desbalances, cuántos atrapé
for t in [0.5, 0.2]:
    al = (p_julio >= t).astype(int)
    print(f"t = {t}: alertas {al.mean():.1%}   precision {precision_score(julio.y, al):.2f}   recall {recall_score(julio.y, al):.2f}")
# sale: t = 0.5: alertas 4.3%   precision 0.64   recall 0.19
#       t = 0.2: alertas 20.4%   precision 0.37   recall 0.52